### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="seismic_bumps",
    dataset_year="2013",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5W902",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/seismic_bumps/ && wget -P local-data-warehouse/seismic_bumps/ https://archive.ics.uci.edu/static/public/266/seismic+bumps.zip && unzip local-data-warehouse/seismic_bumps/seismic+bumps.zip -d local-data-warehouse/seismic_bumps/
""",
    # References
    academic_reference_bibtex=r"""@article{sikora2010application,
  title={Application of rule induction algorithms for analysis of data collected by seismic hazard monitoring systems in coal mines},
  author={Sikora, Marek and Wr{\'o}bel, {\L}ukasz},
  journal={Archives of Mining Sciences},
  volume={55},
  number={1},
  pages={91--114},
  year={2010}
}
""",
    academic_reference_bibtex_key="sikora2010application",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature and reversed the values' ordinal encoding.
- We drop the constant columns "nbumps6", "nbumps7", and "nbumps89".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="HighEnergySeismicBump",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="HighEnergySeismicBump",
)

## Preprocessing

In [ ]:
import arff
import pandas as pd

with open(f"{dataset_mold.path}/seismic-bumps.arff") as f:
    data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])
target_feature = "HighEnergySeismicBump"
df = df.rename(columns={"class": target_feature})
df[target_feature] = df[target_feature].map({"1": "Yes", "0": "No"})

# Drop constant columns
df = df.loc[:, (df != df.iloc[0]).any()]

cat_features = [
    "seismic",
    "seismoacoustic",
    "shift",
    "ghazard",
    "HighEnergySeismicBump",
]

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()